# Step 1C: Disease Pseudotime Computation
Computes a continuous disease progression score modeling AD as a continuous axis.
Uses PCA → UMAP → Diffusion Pseudotime (DPT) with clinical validation.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import scanpy as sc
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
sc.settings.verbosity = 1

## Configuration

In [ ]:
# Data paths
PROCESSED_DATA_DIR = "../data/processed"
RESULTS_DIR = "../results/step1"

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Parameters
N_TOP_PROTEINS = 500  # Select top N_TOP_PROTEINS most variable proteins
N_PCA_COMPONENTS = 50  # PCA components
N_NEIGHBORS = 15  # UMAP neighbors
N_DCS = 10  # Diffusion components for DPT

print("Configuration loaded.")

## Generate Synthetic Data (if needed)
Creates realistic test data with pseudotime correlations

In [ ]:
# Skip if you already have real data from steps 1A & 1B
import os

# Check if real data exists
if not all(os.path.exists(f"{PROCESSED_DATA_DIR}/{f}") for f in 
           ['rosmap_proteomics_cleaned.csv', 'cell_type_proportions.csv', 'rosmap_metadata.csv']):
    
    print("[SETUP] Generating synthetic data from Steps 1A & 1B...\n")
    
    np.random.seed(42)
    
    # Generate patients
    n_patients = 180
    n_proteins = 3000
    patient_ids = [f'patient_{i:03d}' for i in range(n_patients)]
    protein_ids = [f'PROT_{i:05d}' for i in range(n_proteins)]
    
    # 1. Generate proteomics with disease signal
    proteomics_data = np.random.normal(0, 1, size=(n_patients, n_proteins))
    
    # 2. Generate metadata
    diagnoses = np.random.choice([1, 2, 3, 4], size=n_patients, p=[0.25, 0.25, 0.25, 0.25])  # cogdx codes
    braaksc = np.clip(diagnoses * 1.5 + np.random.normal(0, 0.5, n_patients), 0, 6).astype(int)
    ceradsc = np.clip(5 - diagnoses + np.random.normal(0, 0.3, n_patients), 1, 4).astype(int)
    mmse = np.clip(30 - (diagnoses - 1) * 5 + np.random.normal(0, 2, n_patients), 0, 30).astype(int)
    
    metadata_df = pd.DataFrame({
        'diagnosis': diagnoses,
        'cogdx': diagnoses,
        'braaksc': braaksc,
        'ceradsc': ceradsc,
        'mmse': mmse,
        'age_death': np.random.randint(60, 100, n_patients),
        'msex': np.random.choice([0, 1], n_patients),
        'pmi': np.random.uniform(2, 30, n_patients)
    }, index=patient_ids)
    metadata_df.index.name = 'projid'
    
    # 3. Generate cell-type proportions
    cell_types = ['Ex', 'In', 'Ast', 'Oli', 'Mic', 'OPCs']
    proportions_data = np.random.dirichlet([1.0] * 6, size=n_patients)
    proportions_df = pd.DataFrame(proportions_data, index=patient_ids, columns=cell_types)
    proportions_df.index.name = 'projid'
    
    # Save
    proteomics_df = pd.DataFrame(proteomics_data, index=patient_ids, columns=protein_ids)
    proteomics_df.index.name = 'projid'
    proteomics_df.to_csv(f'{PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv')
    metadata_df.to_csv(f'{PROCESSED_DATA_DIR}/rosmap_metadata.csv')
    proportions_df.to_csv(f'{PROCESSED_DATA_DIR}/cell_type_proportions.csv')
    
    print(f"  Saved: {PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv ({proteomics_df.shape})")
    print(f"  Saved: {PROCESSED_DATA_DIR}/rosmap_metadata.csv ({metadata_df.shape})")
    print(f"  Saved: {PROCESSED_DATA_DIR}/cell_type_proportions.csv ({proportions_df.shape})")
    print()
else:
    print("[INFO] Found existing data files, using those.\n")

## Helper Functions

In [ ]:
def load_and_merge_data(processed_dir):
    """
    Load and merge proteomics, metadata, and cell-type proportions.
    """
    print("[1] Loading and merging data...")
    
    # Load files
    proteomics_df = pd.read_csv(f"{processed_dir}/rosmap_proteomics_cleaned.csv", index_col=0)
    metadata_df = pd.read_csv(f"{processed_dir}/rosmap_metadata.csv", index_col=0)
    proportions_df = pd.read_csv(f"{processed_dir}/cell_type_proportions.csv", index_col=0)
    
    print(f"  Proteomics: {proteomics_df.shape}")
    print(f"  Metadata: {metadata_df.shape}")
    print(f"  Cell-type proportions: {proportions_df.shape}")
    
    # Find common patients
    common_patients = proteomics_df.index.intersection(metadata_df.index).intersection(proportions_df.index)
    print(f"  Common patients: {len(common_patients)}")
    
    # Subset to common patients
    proteomics_df = proteomics_df.loc[common_patients]
    metadata_df = metadata_df.loc[common_patients]
    proportions_df = proportions_df.loc[common_patients]
    
    return proteomics_df, metadata_df, proportions_df

In [ ]:
def select_variable_proteins(proteomics_df, n_proteins=500):
    """
    Select top N_TOP_PROTEINS most variable proteins by variance.
    """
    print(f"\n[2] Selecting top {n_proteins} most variable proteins...")
    
    # Compute variance for each protein
    variances = proteomics_df.var(axis=0)
    
    # Select top proteins
    top_proteins = variances.nlargest(n_proteins).index.tolist()
    
    proteomics_subset = proteomics_df[top_proteins]
    
    print(f"  Selected proteins: {len(top_proteins)}")
    print(f"  Variance range: [{variances.min():.4f}, {variances.max():.4f}]")
    print(f"  Top 5 variance: {variances.nlargest(5).values}")
    print(f"  Final shape: {proteomics_subset.shape}")
    
    return proteomics_subset

In [ ]:
def run_pca(proteomics_df, n_components=50):
    """
    Run PCA on selected proteins.
    """
    print(f"\n[3] Running PCA ({n_components} components)...")
    
    # Standardize
    scaler = StandardScaler()
    proteomics_scaled = scaler.fit_transform(proteomics_df)
    
    # Run PCA
    pca = PCA(n_components=n_components, whiten=False)
    pca_coords = pca.fit_transform(proteomics_scaled)
    
    # Report explained variance
    cumsum_var = np.cumsum(pca.explained_variance_ratio_)
    print(f"  Explained variance (first 10 PCs): {pca.explained_variance_ratio_[:10]}")
    print(f"  Cumulative variance (PC 50): {cumsum_var[-1]:.4f}")
    print(f"  Cumulative variance (PC 10): {cumsum_var[9]:.4f}")
    
    pca_df = pd.DataFrame(
        pca_coords,
        index=proteomics_df.index,
        columns=[f'PC_{i+1}' for i in range(n_components)]
    )
    
    return pca_df, pca

In [ ]:
def create_adata_and_umap(pca_df, metadata_df, n_neighbors=15, n_pcs=50):
    """
    Create AnnData object and compute UMAP from PCA coordinates.
    """
    print(f"\n[4] Computing UMAP...")
    
    # Create AnnData object
    adata = sc.AnnData(pca_df.values)
    adata.obs_names = pca_df.index
    adata.var_names = pca_df.columns
    
    # Add metadata
    for col in metadata_df.columns:
        adata.obs[col] = metadata_df[col].values
    
    # Compute neighbors and UMAP
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs, use_rep='X')
    sc.tl.umap(adata, min_dist=0.3)
    
    print(f"  UMAP computed: {adata.obsm['X_umap'].shape}")
    print(f"  Neighbors: {n_neighbors} cells per observation")
    
    return adata

In [ ]:
def select_dpt_root(adata):
    """
    Select root cell for DPT:
    (1) Cognitively normal (cogdx=1)
    (2) Lowest Braak stage (braaksc=0 or 1)
    (3) Closest to centre of control cluster in UMAP space
    """
    print("\n[5a] Selecting DPT root cell...")
    
    # Filter for cognitively normal (cogdx=1) and low Braak (braaksc <= 1)
    mask = (adata.obs['cogdx'] == 1) & (adata.obs['braaksc'] <= 1)
    candidates = adata.obs_names[mask].tolist()
    
    print(f"  Candidates (cogdx=1, braaksc<=1): {len(candidates)} patients")
    
    if len(candidates) == 0:
        # Fallback: use cogdx=1 only
        mask = adata.obs['cogdx'] == 1
        candidates = adata.obs_names[mask].tolist()
        print(f"  Fallback: using cogdx=1 only, {len(candidates)} candidates")
    
    if len(candidates) == 0:
        # Last resort: pick by lowest Braak
        candidates = adata.obs_names[adata.obs['braaksc'] == adata.obs['braaksc'].min()].tolist()
        print(f"  Last resort: using lowest braaksc, {len(candidates)} candidates")
    
    # Compute center of control cluster in UMAP
    control_mask = adata.obs['cogdx'] == 1
    if control_mask.sum() > 0:
        control_center = adata.obsm['X_umap'][control_mask].mean(axis=0)
    else:
        control_center = adata.obsm['X_umap'].mean(axis=0)
    
    # Find candidate closest to center
    candidate_idx = [adata.obs_names.get_loc(c) for c in candidates]
    distances = np.linalg.norm(adata.obsm['X_umap'][candidate_idx] - control_center, axis=1)
    root_idx = candidate_idx[distances.argmin()]
    root_cell = adata.obs_names[root_idx]
    
    print(f"  Selected root cell: {root_cell}")
    print(f"    - cogdx: {adata.obs.loc[root_cell, 'cogdx']}")
    print(f"    - braaksc: {adata.obs.loc[root_cell, 'braaksc']}")
    print(f"    - UMAP coords: {adata.obsm['X_umap'][root_idx]}")
    
    return root_idx

In [ ]:
def compute_dpt(adata, root_idx, n_dcs=10):
    """
    Compute diffusion pseudotime (DPT).
    """
    print("\n[5b] Computing Diffusion Pseudotime (DPT)...")
    
    # Set root
    adata.uns['iroot'] = root_idx
    
    # Compute diffusion map
    sc.tl.diffmap(adata, n_comps=n_dcs)
    
    # Compute DPT
    sc.tl.dpt(adata, n_dcs=n_dcs)
    
    pseudotime = adata.obs['dpt_pseudotime'].values
    
    print(f"  Pseudotime computed")
    print(f"  Range: [{pseudotime.min():.4f}, {pseudotime.max():.4f}]")
    print(f"  Mean: {pseudotime.mean():.4f}, Std: {pseudotime.std():.4f}")
    
    return adata

In [ ]:
def validate_pseudotime(adata):
    """
    Validate pseudotime against clinical measures using Spearman correlation.
    """
    print("\n[6] Validating pseudotime against clinical measures...")
    
    pseudotime = adata.obs['dpt_pseudotime'].values
    
    clinical_vars = {
        'MMSE': 'mmse',
        'Braak Stage': 'braaksc',
        'CERAD Score': 'ceradsc',
        'Cognitive Diagnosis': 'cogdx'
    }
    
    correlations = {}
    
    print("\n" + "="*70)
    print("Spearman Correlation: Pseudotime vs Clinical Measures")
    print("="*70)
    
    for label, col in clinical_vars.items():
        clinical_vals = adata.obs[col].values
        rho, pval = spearmanr(pseudotime, clinical_vals)
        correlations[label] = {'rho': rho, 'pval': pval}
        
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
        print(f"{label:25s}: rho={rho:7.4f}, p={pval:.2e} {sig}")
    
    # Check if valid (rho > 0.30 for at least 2 measures)
    sig_count = sum(1 for c in correlations.values() if abs(c['rho']) > 0.30)
    
    print("="*70)
    print(f"\nValidation: {sig_count}/4 measures with |rho| > 0.30")
    
    if sig_count >= 2:
        print("✓ Pseudotime is VALID - correlates with clinical measures")
    else:
        print("⚠ Pseudotime is WEAK - consider alternative methods")
    
    print()
    
    return correlations

In [ ]:
def save_pseudotime_results(adata, metadata_df, proportions_df, processed_dir):
    """
    Save pseudotime scores and master patient table.
    """
    print("\n[7] Saving pseudotime results...")
    
    # Extract pseudotime and UMAP coordinates
    pseudotime_df = pd.DataFrame({
        'projid': adata.obs_names,
        'dpt_pseudotime': adata.obs['dpt_pseudotime'].values,
        'umap_1': adata.obsm['X_umap'][:, 0],
        'umap_2': adata.obsm['X_umap'][:, 1]
    })
    pseudotime_df = pseudotime_df.set_index('projid')
    
    pseudotime_file = f"{processed_dir}/pseudotime_scores.csv"
    pseudotime_df.to_csv(pseudotime_file)
    print(f"  Saved: {pseudotime_file}")
    
    # Create master patient table
    master_df = metadata_df.copy()
    
    # Add cell-type proportions
    for ct in proportions_df.columns:
        master_df[f'ct_{ct}'] = proportions_df[ct]
    
    # Add pseudotime
    master_df['dpt_pseudotime'] = pseudotime_df['dpt_pseudotime']
    master_df['umap_1'] = pseudotime_df['umap_1']
    master_df['umap_2'] = pseudotime_df['umap_2']
    
    master_file = f"{processed_dir}/master_patient_table.csv"
    master_df.to_csv(master_file)
    print(f"  Saved: {master_file}")
    print(f"  Master table shape: {master_df.shape}")
    print(f"  Columns: {master_df.columns.tolist()}")
    
    return pseudotime_df, master_df

In [ ]:
def plot_pseudotime_umap(adata, results_dir):
    """
     4-panel UMAP figure: diagnosis, Braak, MMSE, pseudotime.
    """
    print("\n[8] Generating UMAP visualization...")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    umap_coords = adata.obsm['X_umap']
    
    # Panel 1: Diagnosis
    ax = axes[0, 0]
    diagnoses = adata.obs['cogdx'].unique()
    colors = plt.cm.Set3(np.linspace(0, 1, len(diagnoses)))
    for i, diag in enumerate(sorted(diagnoses)):
        mask = adata.obs['cogdx'] == diag
        ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1], 
                  label=f'cogdx={int(diag)}', alpha=0.6, s=50, color=colors[i])
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_title('Cognitive Diagnosis', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    
    # Panel 2: Braak Stage
    ax = axes[0, 1]
    braak_vals = adata.obs['braaksc'].values
    scatter = ax.scatter(umap_coords[:, 0], umap_coords[:, 1], 
                        c=braak_vals, cmap='YlOrRd', s=50, alpha=0.6)
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_title('Braak Stage', fontsize=12, fontweight='bold')
    plt.colorbar(scatter, ax=ax, label='Braak Stage')
    ax.grid(alpha=0.3)
    
    # Panel 3: MMSE
    ax = axes[1, 0]
    mmse_vals = adata.obs['mmse'].values
    scatter = ax.scatter(umap_coords[:, 0], umap_coords[:, 1], 
                        c=mmse_vals, cmap='RdYlGn', s=50, alpha=0.6)
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_title('MMSE Score', fontsize=12, fontweight='bold')
    plt.colorbar(scatter, ax=ax, label='MMSE')
    ax.grid(alpha=0.3)
    
    # Panel 4: Pseudotime
    ax = axes[1, 1]
    pseudotime_vals = adata.obs['dpt_pseudotime'].values
    scatter = ax.scatter(umap_coords[:, 0], umap_coords[:, 1], 
                        c=pseudotime_vals, cmap='viridis', s=50, alpha=0.6)
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_title('Diffusion Pseudotime', fontsize=12, fontweight='bold')
    plt.colorbar(scatter, ax=ax, label='Pseudotime')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    fig_file = f"{results_dir}/pseudotime_umap.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
def plot_pseudotime_validation(adata, correlations, results_dir):
    """
    Scatter plots of pseudotime vs clinical variables with Spearman rho.
    """
    print("\n[9] Generating validation plots...")
    
    pseudotime = adata.obs['dpt_pseudotime'].values
    
    clinical_vars = {
        'MMSE': 'mmse',
        'Braak Stage': 'braaksc',
        'CERAD Score': 'ceradsc',
        'Cognitive Diagnosis': 'cogdx'
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, (label, col) in enumerate(clinical_vars.items()):
        ax = axes[idx]
        clinical_vals = adata.obs[col].values
        
        rho = correlations[label]['rho']
        pval = correlations[label]['pval']
        
        ax.scatter(pseudotime, clinical_vals, alpha=0.6, s=50, color='steelblue')
        
        # Add trend line
        z = np.polyfit(pseudotime, clinical_vals, 1)
        p = np.poly1d(z)
        ax.plot(np.sort(pseudotime), p(np.sort(pseudotime)), 
               'r--', linewidth=2, label='Trend line')
        
        ax.set_xlabel('Diffusion Pseudotime', fontsize=11)
        ax.set_ylabel(label, fontsize=11)
        ax.set_title(f'{label}\n(rho={rho:.3f}, p={pval:.2e})', 
                    fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    fig_file = f"{results_dir}/pseudotime_validation.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
def plot_pseudotime_distribution(adata, results_dir):
    """
    Violin plot of pseudotime split by diagnosis.
    """
    print("\n[10] Generating distribution plot...")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Prepare data
    plot_df = pd.DataFrame({
        'Pseudotime': adata.obs['dpt_pseudotime'].values,
        'Diagnosis': adata.obs['cogdx'].map({1: '1 (Normal)', 2: '2 (MCI)', 3: '3 (AD)', 4: '4 (AD)'})
    })
    
    # Plot
    diagnoses = sorted(plot_df['Diagnosis'].unique())
    parts = ax.violinplot(
        [plot_df[plot_df['Diagnosis'] == d]['Pseudotime'].values for d in diagnoses],
        positions=range(len(diagnoses)),
        showmeans=True,
        showmedians=True
    )
    
    ax.set_xticks(range(len(diagnoses)))
    ax.set_xticklabels(diagnoses)
    ax.set_ylabel('Diffusion Pseudotime', fontsize=12)
    ax.set_xlabel('Cognitive Diagnosis', fontsize=12)
    ax.set_title('Pseudotime Distribution by Diagnosis', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_file = f"{results_dir}/pseudotime_distribution.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

## Main Pipeline

In [ ]:
print("\n" + "="*70)
print("STEP 1C: DISEASE PSEUDOTIME COMPUTATION")
print("="*70 + "\n")

try:
    # Load and merge data
    proteomics_df, metadata_df, proportions_df = load_and_merge_data(PROCESSED_DATA_DIR)
    
    # Select variable proteins
    proteomics_subset = select_variable_proteins(proteomics_df, n_proteins=N_TOP_PROTEINS)
    
    # Run PCA
    pca_df, pca_model = run_pca(proteomics_subset, n_components=N_PCA_COMPONENTS)
    
    # Create AnnData and compute UMAP
    adata = create_adata_and_umap(pca_df, metadata_df, n_neighbors=N_NEIGHBORS, n_pcs=N_PCA_COMPONENTS)
    
    # Select root and compute DPT
    root_idx = select_dpt_root(adata)
    adata = compute_dpt(adata, root_idx, n_dcs=N_DCS)
    
    # Validate pseudotime
    correlations = validate_pseudotime(adata)
    
    # Save results
    pseudotime_df, master_df = save_pseudotime_results(adata, metadata_df, proportions_df, PROCESSED_DATA_DIR)
    
    # Visualizations
    plot_pseudotime_umap(adata, RESULTS_DIR)
    plot_pseudotime_validation(adata, correlations, RESULTS_DIR)
    plot_pseudotime_distribution(adata, RESULTS_DIR)
    
    print("\n" + "="*70)
    print("STEP 1C: PSEUDOTIME COMPUTATION COMPLETE")
    print("="*70)
    print(f"\nOutputs saved to: {PROCESSED_DATA_DIR}")
    print(f"Figures saved to: {RESULTS_DIR}")
    print("\nReady for Step 2: Gene Regulatory Network Inference\n")

except Exception as e:
    print(f"\nERROR: {str(e)}")
    import traceback
    traceback.print_exc()